# Music Albums Domain — Benchmark Record Generator

Domain-specific pools, templates, generation functions, and query builders for the music albums benchmark domain.
Produces `BenchmarkExample` records for L1–L5 complexity levels covering genre, country, performer, and record-label constraints.

**Outputs:**
- `generate_music_albums_example(complexity, idx, rng)` — callable used by the master generation loop
- Pool DataFrames: `ma_genres_df`, `ma_countries_df`, `ma_performers_df`, `ma_labels_df`

## 1. Configuration

Load shared helpers from `common_helpers.py` when the notebook is executed in standalone mode.

In [ ]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which requires the nbformat package.
from pathlib import Path
if "BenchmarkExample" not in globals():
    exec(Path("common_helpers.py").read_text(encoding="utf-8"), globals())


## 2. Pool building

Fetch or load cached Wikidata value pools for genres, countries, performers, and record labels.
Common high-frequency values are also kept as in-memory lists for faster random sampling.

In [ ]:
Q_MUSIC_ALBUM = ensure_qid("музыкальный альбом", fallback_qid="Q482994")

ma_genres_df = load_or_build_pool("music_albums_genres_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P136", "genre", limit=350))
ma_countries_df = load_or_build_pool("music_albums_countries_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P495", "country", limit=250))
ma_performers_df = load_or_build_pool("music_albums_performers_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P175", "performer", limit=350))
ma_labels_df = load_or_build_pool("music_albums_labels_ru", lambda: build_value_pool_ru(Q_MUSIC_ALBUM, "P264", "label", limit=250))

COMMON_MA_GENRES_RU = ["рок", "поп-музыка", "джаз", "хип-хоп", "классическая музыка", "электронная музыка", "метал"]
COMMON_COUNTRIES_RU = ["США", "Россия", "Великобритания", "Франция", "Германия", "Япония", "Канада"]

def pick_music_constraints(rng: random.Random):
    if rng.random() < 0.6:
        g_ru = rng.choice(COMMON_MA_GENRES_RU)
        c_ru = rng.choice(COMMON_COUNTRIES_RU)
        g_qid = ensure_qid(g_ru)
        c_qid = ensure_qid(c_ru)
    else:
        g_qid, g_ru = pick_from_df(ma_genres_df, "genre_qid", "genreLabelRu", rng)
        c_qid, c_ru = pick_from_df(ma_countries_df, "country_qid", "countryLabelRu", rng)
    perf_qid, perf_ru = pick_from_df(ma_performers_df, "performer_qid", "performerLabelRu", rng)
    lab_qid, lab_ru = pick_from_df(ma_labels_df, "label_qid", "labelLabelRu", rng)
    return g_qid, g_ru, c_qid, c_ru, perf_qid, perf_ru, lab_qid, lab_ru

## 3. SPARQL query helpers

Build and execute SPARQL WHERE clauses for music album retrieval, filtering by genre, country, performer, label, and release year range.

In [ ]:
def run_music_album_query(
    genre_qid: str,
    country_qid: str,
    performer_qid: Optional[str],
    label_qid: Optional[str],
    y1: int,
    y2: int,
    limit: int = 250,
):
    where = [
        f"?item wdt:P136 wd:{genre_qid} .",
        f"?item wdt:P495 wd:{country_qid} .",
        "?item wdt:P577 ?date .",
        "BIND(YEAR(?date) AS ?year) .",
        f"FILTER(?year >= {int(y1)} && ?year <= {int(y2)}) .",
    ]
    if performer_qid:
        where.append(f"?item wdt:P175 wd:{performer_qid} .")
    if label_qid:
        where.append(f"?item wdt:P264 wd:{label_qid} .")
    sparql, items = select_items_with_ru_label(Q_MUSIC_ALBUM, where, limit=limit, item_var="item")
    return sparql, items, where

## 4. Natural language generation

Produce a Russian-language question string from structured constraint values.

In [ ]:
def nlg_music_album_ru(genre_ru: str, country_ru: str, performer_ru: Optional[str], label_ru: Optional[str], y1: int, y2: int, k: int) -> str:
    cond = [f"жанра «{genre_ru}»", f"из страны: {country_ru}"]
    if performer_ru:
        cond.append(f"исполнитель: {performer_ru}")
    if label_ru:
        cond.append(f"лейбл: {label_ru}")
    cond.append(f"в период {y1}–{y2} годов" if y1 != y2 else f"{y1} года")
    return f"Назови {k} музыкальных альбомов, " + ", ".join(cond) + "."

## 5. Dataset generation

Main entry point `generate_music_albums_example` samples constraints, executes the SPARQL query, and returns a `BenchmarkExample` with gold answers and an ASK-based validator.

In [ ]:
def generate_music_albums_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 90) -> BenchmarkExample:
    for _ in range(max_attempts):
        g_qid, g_ru, c_qid, c_ru, perf_qid, perf_ru, lab_qid, lab_ru = pick_music_constraints(rng)

        decade = rng.choice([(1970,1979),(1980,1989),(1990,1999),(2000,2009),(2010,2019)])
        y1, y2 = decade
        performer_qid = None
        performer_ru = None
        label_qid = None
        label_ru = None
        k = 5

        if complexity == "L1":
            pass
        elif complexity == "L2":
            performer_qid, performer_ru = perf_qid, perf_ru
        elif complexity == "L3":
            performer_qid, performer_ru = perf_qid, perf_ru
            label_qid, label_ru = lab_qid, lab_ru
        elif complexity == "L4":
            performer_qid, performer_ru = perf_qid, perf_ru
            label_qid, label_ru = lab_qid, lab_ru
            y1, y2 = rng.choice([(1970,1974),(1980,1984),(1990,1994),(2000,2004),(2010,2014)])
            k = 12
        elif complexity == "L5":
            performer_qid, performer_ru = perf_qid, perf_ru
            label_qid, label_ru = lab_qid, lab_ru
            y1, y2 = 2500, 2500
            k = 5
        else:
            raise ValueError(f"Unknown complexity: {complexity}")

        sparql, items, where = run_music_album_query(g_qid, c_qid, performer_qid, label_qid, y1, y2, limit=300)

        gold = items[:300]
        gold_qids = [q for q,_ in gold]
        gold_lbls = [l for _,l in gold]
        truncated = len(items) >= 300
        ask = build_ask_validator(Q_MUSIC_ALBUM, where, item_var="item")

        return BenchmarkExample(
            id=f"music_albums_{complexity.lower()}_{idx:04d}",
            domain="music_albums",
            complexity=complexity,
            query_text_ru=nlg_music_album_ru(g_ru, c_ru, performer_ru, label_ru, y1, y2, k),
            constraints={
                "genre_qid": g_qid, "genre_ru": g_ru,
                "country_qid": c_qid, "country_ru": c_ru,
                "performer_qid": performer_qid, "performer_ru": performer_ru,
                "label_qid": label_qid, "label_ru": label_ru,
                "year_from": y1, "year_to": y2,
            },
            requested_count=k,
            gold_answer_qids=gold_qids,
            gold_answer_labels_ru=gold_lbls,
            sparql_query=sparql,
            created_at=utc_now_z(),
            gold_truncated=truncated,
            ask_validator_sparql=ask,
        )

    raise RuntimeError(f"Failed to generate music_albums example {complexity} after {max_attempts} attempts")

## 6. Sanity check

Quick smoke test to confirm the generator runs end-to-end. Failures are caught and surfaced as warnings so that downstream orchestration notebooks can still import this domain.

In [ ]:
try:
    _test_ma = generate_music_albums_example("L1", 0, random.Random(1))
except Exception as e:
    print("[WARN] sanity music_albums skipped:", e)
if 'DOMAIN_PAUSE_S' in globals() and DOMAIN_PAUSE_S and DOMAIN_PAUSE_S > 0:
    time.sleep(float(DOMAIN_PAUSE_S))